# MADAR — Evaluation of Vehicle Detection, Plate Detection, and OCR
Use a T4 GPU runtime. Run cells in order. This notebook does not train models and does not evaluate tracking.

## Outputs
- Detection: Precision, Recall, F1, mAP@50, mAP@50–95, PR/F1 curves, confusion matrices, and prediction/label examples.
- OCR on ground-truth crops: exact text accuracy, Character Error Rate (CER), aligned-character micro precision/recall/F1, and a character confusion matrix including insertion/deletion gaps.
- JSON/CSV metrics, dataset manifests, package versions, model checksums, and a downloadable results ZIP.

Detection summary P/R and F1 use Ultralytics' max-F1 operating point. They are not necessarily the values implied by the plotted detection confusion matrix. Confidence scores are not accuracy. Detection matrices include a background row/column for false positives and false negatives; there is no count of true-negative background objects.

## Data requirements
1. Each detector needs its own YOLO detection dataset: images, normalized `class_id x_center y_center width height` labels, and data.yaml. Use an independent test split. Missing label files cause an error; intentionally negative images require an empty label file.
2. OCR needs correct text per crop, not just plate bounding boxes. A crop/template preparation cell is provided. Enter the visible English/ASCII plate text in its ground_truth column and upload the completed CSV. Do not use model predictions as ground truth.
3. Do not use images from training, validation/model selection, or augmented duplicates as independent test samples. Check original dataset provenance; an exported test folder alone does not establish independence. Neighboring video frames must not leak across train and test.
4. The uploaded training notebook used CARPK test as validation, so that split is not a new independent test for those vehicle weights.

This notebook has been syntax-checked; real model evaluation requires the dataset uploads below. No evaluation scores are prefilled.


## Dataset selection
| Source | Appropriate use | Preparation |
|---|---|---|
| Saudi Licence Plates, Roboflow | Saudi plate detection; recommended starting point | Export v1 as YOLOv8, use test split, map LP to the model plate class. v1 reports 118 test images and no augmentation. Text transcripts have not been verified. |
| License Plate Project, test-vaxvp | Additional plate-detection evaluation | Export YOLOv8, use test split, verify provenance and domain. The published Roboflow scores belong to its hosted model, not MADAR. |
| license-plates-wbvpr, yassine-bouaza | Not verified | The page could not be inspected. Upload its export so classes/labels can be checked. |
| Dragon Lake Parking (DLP) | Aerial parking vehicle detection | Raw video + ground truth need parsing, time alignment, and conversion from geographic/oriented annotations to pixel boxes. Include stationary obstacle vehicles as well as moving vehicle agents. JSON trajectories alone are not a ready YOLO image dataset. This notebook accepts the converted YOLO export, not native DLP files. |

DLP raw data is about 168 GB; start with its sample. The authors restrict use to non-commercial purposes and prohibit redistribution. Follow their terms for your intended use.

Sources: [Saudi v1](https://universe.roboflow.com/cars-o2wwf/saudi-licence-plates/dataset/1), [License Plate Project](https://universe.roboflow.com/test-vaxvp/license-plate-project-adaad/dataset/1), [DLP](https://sites.google.com/berkeley.edu/dlp-dataset).


## 1. Install dependencies
Run this in a dedicated evaluation session. Optional LangChain packages are removed because they caused the observed PaddleX import conflict. If Colab requests a restart, restart the session before continuing.

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "ultralytics==8.4.174", "paddlepaddle==3.2.2", "paddleocr==3.3.2",
    "pandas", "matplotlib", "pyyaml"])
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y",
                       "langchain", "langchain-community"])


0

## 2. Upload the latest MADAR project ZIP
The latest ZIP includes both detection weights and OCR inference files.

In [2]:
from google.colab import files
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, Image
import os, zipfile, shutil, json, math, csv, hashlib, re
import numpy as np, pandas as pd, cv2, torch, yaml
import matplotlib.pyplot as plt
from collections import Counter
from ultralytics import YOLO

ROOT = Path("/content/madar_evaluation")
MODEL_ROOT = ROOT / "models"
DATA_ROOT = ROOT / "datasets"
OUT = ROOT / "results" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%f")
for p in [MODEL_ROOT, DATA_ROOT, OUT]:
    p.mkdir(parents=True, exist_ok=True)

def extract_zip(path, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(path) as archive:
        for entry in archive.infolist():
            target = (destination / entry.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError("Unsafe ZIP path")
            if ((entry.external_attr >> 16) & 0o170000) == 0o120000:
                raise ValueError("ZIP symlinks are not supported")
        archive.extractall(destination)

for name,content in files.upload().items():
    target = MODEL_ROOT / Path(name).name
    target.write_bytes(content)
    if target.suffix.lower() == ".zip":
        extract_zip(target, MODEL_ROOT)

def unique_file(name):
    matches = list(MODEL_ROOT.rglob(name))
    assert len(matches) == 1, f"Expected one {name}; found {len(matches)}"
    return matches[0]

VEHICLE_WEIGHTS = unique_file("MADAR_vehicle_final.pt")
PLATE_WEIGHTS = unique_file("MADAR_license_plate_final.pt")
OCR_DIR = unique_file("inference.yml").parent
assert all((OCR_DIR / n).is_file() for n in ["inference.json", "inference.pdiparams"])
vehicle_model = YOLO(str(VEHICLE_WEIGHTS))
plate_model = YOLO(str(PLATE_WEIGHTS))
print("Vehicle model classes:",vehicle_model.names)
print("Plate model classes:",plate_model.names)
DEVICE = 0 if torch.cuda.is_available() else "cpu"


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


Saving MADAR_Project-20261006T155911Z-1-001.zip to MADAR_Project-20261006T155911Z-1-001.zip
Vehicle model classes: {0: 'vehicle'}
Plate model classes: {0: 'LP'}


## 3. Upload labeled detection dataset ZIPs
Download Roboflow data in YOLOv8 format, not image-only format. You can upload plate and vehicle exports together. A plate-only dataset cannot evaluate vehicle detection even when cars are visible, because their vehicle boxes are not labeled.

In [ ]:
for name,content in files.upload().items():
    target = DATA_ROOT / Path(name).name
    target.write_bytes(content)
    if target.suffix.lower() == ".zip":
        extract_zip(target, DATA_ROOT / target.stem)
yaml_paths = sorted(set(DATA_ROOT.rglob("data.yaml")) | set(DATA_ROOT.rglob("data.yml")))
assert yaml_paths, "No data.yaml found. Native DLP JSON is not supported; convert to YOLO first."
for p in yaml_paths:
    config = yaml.safe_load(p.read_text())
    print("\nDATA YAML:",p)
    print("Classes:",config.get("names"))
    print("Splits:",{k:config.get(k) for k in ["train","val","test"]})


## 4. Choose datasets and class mappings
Paste the paths printed above. Leave a model's YAML blank only if its dataset is unavailable; it will be reported as not evaluated, not as zero accuracy.

CLASS_MAP maps dataset class IDs to model class IDs. For a single plate class LP=0 and single model class=0, use `{0: 0}`. If a dataset has car=0, pedestrian=1 and the model has vehicle=0, use `{0: 0, 1: None}` to explicitly ignore pedestrian annotations while retaining all images. Every dataset class must have an explicit mapping. Only merge classes that genuinely belong to the modeled concept; do not map plate boxes to vehicle boxes.

Keep test as the split. Change to val only for an explicitly labeled validation result; do not call it an independent test result. Choose settings before looking at test scores; use a separate validation set for tuning.


In [ ]:
PLATE_DATA_YAML = ""  # Paste the plate dataset data.yaml path here.
VEHICLE_DATA_YAML = ""  # Paste a converted aerial vehicle dataset data.yaml path here.
PLATE_SPLIT = "test"
VEHICLE_SPLIT = "test"
PLATE_CLASS_MAP = {0: 0}
VEHICLE_CLASS_MAP = {0: 0}

PLATE_SOURCE_URL = "https://universe.roboflow.com/cars-o2wwf/saudi-licence-plates/dataset/1"
VEHICLE_SOURCE_URL = "https://sites.google.com/berkeley.edu/dlp-dataset"
# Update these source descriptions if you use different exports.

EVAL_IMAGE_SIZE = 1280
EVAL_BATCH = 4
MAX_DETECTIONS = 3000
AP_MIN_CONF = 0.001
NMS_IOU = 0.70
assert PLATE_DATA_YAML or VEHICLE_DATA_YAML, "Set at least one dataset YAML path before evaluation."


## 5. Validate and prepare evaluation data
This copies the selected split into a separate folder and remaps labels without editing the source data. It checks label format, class IDs, box bounds, missing labels, and duplicate image bytes within the split. Duplicate images cause an error rather than silently inflating the score.

In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def canonical_names(names):
    if isinstance(names, list):
        return dict(enumerate(names))
    return {int(k):v for k,v in names.items()}

def parse_label_line(line):
    parts = line.split()
    if len(parts) != 5:
        raise ValueError("Expected YOLO detection labels with exactly 5 values per row")
    values = [float(v) for v in parts]
    if not all(math.isfinite(v) for v in values):
        raise ValueError("Nonfinite label value")
    cid,x,y,w,h = values
    if cid != int(cid) or cid < 0:
        raise ValueError("Invalid class ID")
    if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
        raise ValueError("Invalid normalized bounding box")
    if x-w/2 < -1e-4 or y-h/2 < -1e-4 or x+w/2 > 1.0001 or y+h/2 > 1.0001:
        raise ValueError("Bounding box extends outside image")
    return int(cid),x,y,w,h

def prepare_dataset(kind, yaml_file, split, model, class_map, source_url):
    yp = Path(yaml_file).resolve()
    assert yp.is_file(),f"Missing YAML: {yp}"
    assert split in {"test","val"}, "Only test or explicitly named validation evaluation is allowed"
    config = yaml.safe_load(yp.read_text())
    data_names = canonical_names(config["names"])
    model_names = canonical_names(model.names)
    assert set(class_map) == set(data_names), f"Map every dataset class explicitly: {data_names}"
    assert all(v is None or v in model_names for v in class_map.values()),"Mapping refers to unknown model class"
    assert config.get(split), f"No {split} split in dataset YAML; do not fall back to training"
    raw_base = Path(str(config.get("path", ".")))
    base = raw_base if raw_base.is_absolute() else (yp.parent / raw_base).resolve()
    roots = config[split] if isinstance(config[split],list) else [config[split]]
    image_paths = []
    for entry in roots:
        relative = Path(str(entry))
        candidates = [relative] if relative.is_absolute() else [base/relative,yp.parent/relative]
        # Some exported YAML files use ../test/images even when test is beside data.yaml.
        if not relative.is_absolute():
            candidates.append(yp.parent / str(entry).removeprefix("../"))
        found = next((p.resolve() for p in candidates if p.is_dir()),None)
        assert found is not None, f"Split directory not found: {entry}. Set valid directory paths in data.yaml."
        image_paths.extend(p.resolve() for p in found.rglob("*") if p.suffix.lower() in IMAGE_SUFFIXES)
    image_paths = sorted(set(image_paths))
    assert image_paths,"Selected split is empty"
    prepared = OUT / "prepared" / kind
    (prepared/"test/images").mkdir(parents=True,exist_ok=True)
    (prepared/"test/labels").mkdir(parents=True,exist_ok=True)
    hashes,manifest = {},[]
    annotation_count,ignored_count,negative_count,max_objects = 0,0,0,0
    for index,img in enumerate(image_paths):
        parts = list(img.parts)
        assert "images" in parts,f"Images must be under an images directory: {img}"
        position = len(parts)-1-parts[::-1].index("images")
        parts[position] = "labels"
        label = Path(*parts).with_suffix(".txt")
        assert label.is_file(), f"Missing labels: {label}. Use explicit empty files for true negative images."
        digest = hashlib.sha256(img.read_bytes()).hexdigest()
        assert digest not in hashes,f"Duplicate image bytes: {img} and {hashes.get(digest)}"
        hashes[digest] = str(img)
        assert cv2.imread(str(img)) is not None,f"Unreadable image: {img}"
        output_lines = []
        for line in label.read_text().splitlines():
            if not line.strip():
                continue
            cid,x,y,w,h = parse_label_line(line)
            assert cid in class_map,f"Unknown class {cid} in {label}"
            target_class = class_map[cid]
            if target_class is None:
                ignored_count += 1
                continue
            output_lines.append(f"{target_class} {x:.8f} {y:.8f} {w:.8f} {h:.8f}")
        name = f"{index:07d}{img.suffix.lower()}"
        shutil.copy2(img,prepared/"test/images"/name)
        (prepared/"test/labels"/Path(name).with_suffix(".txt")).write_text("\n".join(output_lines))
        annotation_count += len(output_lines)
        negative_count += not output_lines
        max_objects = max(max_objects,len(output_lines))
        manifest.append({"prepared_image":name,"source_image":str(img),"source_label":str(label),
                         "image_sha256":digest,"retained_annotations":len(output_lines)})
    assert annotation_count > 0,"No target annotations remain after class mapping"
    assert MAX_DETECTIONS >= max_objects,"max_det is lower than an annotated object count"
    prepared_yaml = prepared/"data.yaml"
    prepared_yaml.write_text(yaml.safe_dump({"path":str(prepared),"train":"test/images",
                                            "val":"test/images","test":"test/images",
                                            "names":model_names},sort_keys=False))
    pd.DataFrame(manifest).to_csv(prepared/"dataset_manifest.csv",index=False)
    info = {"kind":kind,"source_url":source_url,"source_yaml":str(yp),"source_split":split,
            "images":len(image_paths),"annotations":annotation_count,"negative_images":negative_count,
            "explicitly_ignored_annotations":ignored_count,"class_mapping":class_map,
            "dataset_classes":data_names,"model_classes":model_names,
            "max_annotations_per_image":max_objects}
    (prepared/"dataset_info.json").write_text(json.dumps(info,indent=2))
    print(kind,info)
    return prepared_yaml,info

prepared_sets = {}
for kind,path,split,model,mapping,url in [
    ("plate",PLATE_DATA_YAML,PLATE_SPLIT,plate_model,PLATE_CLASS_MAP,PLATE_SOURCE_URL),
    ("vehicle",VEHICLE_DATA_YAML,VEHICLE_SPLIT,vehicle_model,VEHICLE_CLASS_MAP,VEHICLE_SOURCE_URL)]:
    if path:
        prepared_sets[kind] = prepare_dataset(kind,path,split,model,mapping,url)
    else:
        print(kind,"NOT EVALUATED: no labeled dataset supplied")


## 6. Evaluate both detectors
Low conf=0.001 retains predictions needed for AP/PR curves. Reported P/R use the library max-F1 operating point; do not compare them directly with deployment confidence=0.10. No tracking or OCR is used in detection evaluation.

In [ ]:
detection_metrics = []
run_dirs = {}
for kind,model in [("plate",plate_model),("vehicle",vehicle_model)]:
    if kind not in prepared_sets:
        detection_metrics.append({"model":kind,"status":"not_evaluated"})
        continue
    prepared_yaml,info = prepared_sets[kind]
    metrics = model.val(data=str(prepared_yaml),split="test",imgsz=EVAL_IMAGE_SIZE,
                        batch=EVAL_BATCH,device=DEVICE,conf=AP_MIN_CONF,iou=NMS_IOU,
                        max_det=MAX_DETECTIONS,plots=True,save_txt=True,save_conf=True,
                        workers=0,project=str(OUT/"detection"),name=kind,exist_ok=False)
    p,r = float(metrics.box.mp),float(metrics.box.mr)
    f1 = float(np.mean(metrics.box.f1)) if len(metrics.box.f1) else 0.0
    row = {"model":kind,"status":"evaluated","source_split":info["source_split"],
           "images":info["images"],"annotations":info["annotations"],
           "precision_max_f1":p,"recall_max_f1":r,"mean_class_f1_max_f1":f1,
           "mAP50":float(metrics.box.map50),"mAP50_95":float(metrics.box.map)}
    detection_metrics.append(row)
    run_dir = Path(metrics.save_dir)
    run_dirs[kind] = run_dir
    (run_dir/"metrics.json").write_text(json.dumps(row,indent=2))
    class_ids = list(metrics.box.ap_class_index)
    per_class = []
    for j,cid in enumerate(class_ids):
        per_class.append({"class_id":int(cid),"class_name":model.names[int(cid)],
                          "precision":float(metrics.box.p[j]),"recall":float(metrics.box.r[j]),
                          "f1":float(metrics.box.f1[j]),"AP50":float(metrics.box.ap50[j]),
                          "AP50_95":float(metrics.box.ap[j])})
    pd.DataFrame(per_class).to_csv(run_dir/"per_class_metrics.csv",index=False)
    matrix = metrics.confusion_matrix.matrix
    names = [model.names[k] for k in sorted(model.names)] + ["background"]
    pd.DataFrame(matrix,index=names,columns=names).to_csv(run_dir/"confusion_matrix_counts.csv")
    print("Confusion CSV: rows=predicted, columns=ground truth")
    for filename in ["confusion_matrix.png","confusion_matrix_normalized.png",
                     "BoxPR_curve.png","BoxF1_curve.png","BoxP_curve.png","BoxR_curve.png",
                     "val_batch0_labels.jpg","val_batch0_pred.jpg"]:
        if (run_dir/filename).exists():
            print(kind,filename)
            display(Image(filename=str(run_dir/filename),width=800))

detection_df = pd.DataFrame(detection_metrics)
detection_df.to_csv(OUT/"detection_metrics.csv",index=False)
display(detection_df)


## 7. Prepare OCR ground-truth crops and a text template
This crops the annotated plate boxes, isolating OCR from plate detector mistakes. The generated CSV has an empty ground_truth column; it is not ready to score until correct text is entered by a person.

Download the ZIP, inspect each crop, enter its visible English/ASCII plate text, and save the CSV as UTF-8. Preserve leading zeros. For Saudi plates, evaluate the English/numeric region separately if that is the model's intended output; do not mix Arabic transcripts into an English-only claim. Multiline crops may need manual text-region cropping. Retain unreadable samples in a separate list rather than inventing text.

Alternatively supply your own ZIP with cropped images and `ocr_ground_truth.csv` containing image,ground_truth. Paths are relative to the CSV's directory. This evaluates recognition on provided crops, not end-to-end plate detection + OCR.


In [ ]:
OCR_INPUT = ROOT/"ocr_inputs"
OCR_INPUT.mkdir(parents=True,exist_ok=True)
if "plate" in prepared_sets:
    prepared_yaml,_ = prepared_sets["plate"]
    prepared = prepared_yaml.parent
    pack = OCR_INPUT/"ground_truth_crops"
    (pack/"crops").mkdir(parents=True,exist_ok=True)
    template = []
    for img_path in sorted((prepared/"test/images").iterdir()):
        img = cv2.imread(str(img_path))
        height,width = img.shape[:2]
        label = prepared/"test/labels"/img_path.with_suffix(".txt").name
        for index,line in enumerate(label.read_text().splitlines()):
            cid,x,y,w,h = parse_label_line(line)
            x1,y1 = max(0,math.floor((x-w/2)*width)),max(0,math.floor((y-h/2)*height))
            x2,y2 = min(width,math.ceil((x+w/2)*width)),min(height,math.ceil((y+h/2)*height))
            assert x2 > x1 and y2 > y1,"Empty ground-truth crop"
            relative = f"crops/{img_path.stem}_{index}.png"
            assert cv2.imwrite(str(pack/relative),img[y1:y2,x1:x2]),"Crop save failed"
            template.append({"image":relative,"ground_truth":"","source_image":img_path.name,
                             "x1":x1,"y1":y1,"x2":x2,"y2":y2})
    template_path = pack/"ocr_ground_truth.csv"
    pd.DataFrame(template).to_csv(template_path,index=False,encoding="utf-8-sig")
    print("Crops requiring manual text labels:",len(template))
    files.download(shutil.make_archive(str(pack),"zip",pack))
else:
    print("No plate test dataset supplied. Upload your own OCR crop dataset in the next cell.")


## 8. Upload completed OCR labels or a labeled crop ZIP
If you edited only the generated CSV, upload it here; the crops remain in the Colab session. For an independent OCR dataset, upload a ZIP containing ocr_ground_truth.csv and its images. Leave the upload empty to skip OCR explicitly.

In [ ]:
ocr_uploads = files.upload()
OCR_MANIFEST = None
for name,content in ocr_uploads.items():
    target = OCR_INPUT/Path(name).name
    target.write_bytes(content)
    if target.suffix.lower() == ".zip":
        destination = OCR_INPUT/target.stem
        extract_zip(target,destination)
        candidates = list(destination.rglob("ocr_ground_truth.csv"))
        assert len(candidates) == 1,"OCR ZIP must contain one ocr_ground_truth.csv"
        OCR_MANIFEST = candidates[0]
    elif target.suffix.lower() == ".csv":
        # An edited generated template keeps paths relative to the generated crop folder.
        generated = OCR_INPUT/"ground_truth_crops"
        destination = generated if (generated/"crops").exists() else OCR_INPUT
        OCR_MANIFEST = destination/"ocr_ground_truth.csv"
        OCR_MANIFEST.write_bytes(content)
print("OCR manifest:",OCR_MANIFEST or "NOT EVALUATED: no completed text labels uploaded")


## 9. OCR scoring and character confusion matrix
Normalization: uppercase and remove whitespace only. No character substitutions, Arabic transliteration, or guessed text are applied. Raw text is also retained. Metrics include raw exact accuracy and normalized exact accuracy.

CER = (substitutions + deletions + insertions) / reference characters. It may exceed 1. Aligned-character micro precision = correct / (correct + substitutions + insertions); recall = correct / (correct + substitutions + deletions). These are OCR character measures, not object-detection precision/recall. The confusion matrix uses ground-truth rows, predicted columns, and <eps> for insertion/deletion gaps. A single deterministic minimum-edit alignment is used.

Blank ground truth or invalid images stop evaluation. OCR runtime errors are retained as failed empty predictions, not silently excluded. Unsupported reference characters are counted and reported.


In [ ]:
EPS = "<eps>"
def normalize_text(text):
    return "".join(str(text).upper().split())

def align_text(reference,prediction):
    m,n = len(reference),len(prediction)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1,m+1):
        for j in range(1,n+1):
            dp[i][j] = min(dp[i-1][j-1]+(reference[i-1]!=prediction[j-1]),
                           dp[i-1][j]+1,dp[i][j-1]+1)
    i,j,pairs = m,n,[]
    while i or j:
        if i and j and dp[i][j] == dp[i-1][j-1]+(reference[i-1]!=prediction[j-1]):
            pairs.append((reference[i-1],prediction[j-1])); i-=1; j-=1
        elif i and dp[i][j] == dp[i-1][j]+1:
            pairs.append((reference[i-1],EPS)); i-=1
        else:
            pairs.append((EPS,prediction[j-1])); j-=1
    pairs.reverse()
    return pairs

ocr_summary = {"model":"ocr","status":"not_evaluated"}
if OCR_MANIFEST:
    from paddleocr import TextRecognition
    ocr_config = yaml.safe_load((OCR_DIR/"inference.yml").read_text())
    ocr = TextRecognition(model_name=ocr_config["Global"]["model_name"],
                          model_dir=str(OCR_DIR),device="cpu",enable_mkldnn=False)
    gt = pd.read_csv(OCR_MANIFEST,dtype=str,keep_default_na=False)
    assert {"image","ground_truth"}.issubset(gt.columns),"Required columns: image,ground_truth"
    assert len(gt)>0,"Empty OCR dataset"
    assert gt["image"].is_unique,"Each OCR crop must have one reference transcription"
    assert gt.ground_truth.map(normalize_text).ne("").all(),"Fill every ground_truth entry before scoring"
    ocr_out = OUT/"ocr"
    ocr_out.mkdir(exist_ok=True)
    allowed = set(ocr_config["PostProcess"]["character_dict"])
    details,all_pairs,totals = [],[],Counter()
    unsupported_refs = 0
    for index,row in gt.iterrows():
        path = (OCR_MANIFEST.parent/row["image"]).resolve()
        assert path.is_relative_to(OCR_MANIFEST.parent.resolve()),"OCR image path escapes dataset root"
        img = cv2.imread(str(path))
        assert img is not None,f"Cannot read OCR image: {path}"
        raw_ref = row["ground_truth"]
        ref = normalize_text(raw_ref)
        unsupported = sorted(set(ref)-allowed)
        unsupported_refs += bool(unsupported)
        prediction,score,error = "",None,""
        try:
            results = list(ocr.predict(input=img,batch_size=1))
            assert results,"No OCR prediction returned"
            payload = results[0].json
            if isinstance(payload,str): payload=json.loads(payload)
            payload=payload.get("res",payload)
            prediction,score=str(payload["rec_text"]),float(payload["rec_score"])
        except Exception as exc:
            error=str(exc)
        pred = normalize_text(prediction)
        pairs=align_text(ref,pred)
        all_pairs.extend(pairs)
        counts=Counter()
        for actual,guess in pairs:
            if actual==EPS: counts["insertions"]+=1
            elif guess==EPS: counts["deletions"]+=1
            elif actual==guess: counts["correct"]+=1
            else: counts["substitutions"]+=1
        totals.update(counts)
        details.append({"image":row["image"],"ground_truth_raw":raw_ref,"prediction_raw":prediction,
                        "reference":ref,"prediction":pred,"confidence":score,
                        "exact_raw":raw_ref==prediction,"exact_normalized":ref==pred,
                        "reference_characters":len(ref),**{k:counts[k] for k in ["correct","substitutions","deletions","insertions"]},
                        "unsupported_reference_characters":"".join(unsupported),"error":error})
        if index%25==0: print("OCR samples:",index+1,"/",len(gt))
    details_df=pd.DataFrame(details)
    correct,subs,dels,ins=[totals[k] for k in ["correct","substitutions","deletions","insertions"]]
    total_chars=int(details_df.reference_characters.sum())
    precision=correct/(correct+subs+ins) if correct+subs+ins else 0.0
    recall=correct/(correct+subs+dels) if correct+subs+dels else 0.0
    ocr_summary={"model":"ocr","status":"evaluated","samples":len(gt),
                 "exact_accuracy_raw":float(details_df.exact_raw.mean()),
                 "exact_accuracy_normalized":float(details_df.exact_normalized.mean()),
                 "CER":(subs+dels+ins)/total_chars,"character_precision":precision,
                 "character_recall":recall,"character_f1":2*precision*recall/(precision+recall) if precision+recall else 0,
                 "correct_characters":correct,"substitutions":subs,"deletions":dels,"insertions":ins,
                 "reference_characters":total_chars,"runtime_error_samples":int(details_df.error.ne("").sum()),
                 "samples_with_unsupported_reference_characters":unsupported_refs,
                 "scope":"recognition on supplied ground-truth crops; not end-to-end detection + OCR"}
    alphabet=sorted({c for pair in all_pairs for c in pair})
    positions={c:i for i,c in enumerate(alphabet)}
    matrix=np.zeros((len(alphabet),len(alphabet)),dtype=int)
    for actual,guess in all_pairs: matrix[positions[actual],positions[guess]]+=1
    matrix_df=pd.DataFrame(matrix,index=alphabet,columns=alphabet)
    matrix_df.to_csv(ocr_out/"character_confusion_matrix_counts.csv")
    # Limit only the preview; the exported CSV contains the complete alphabet.
    frequency=Counter(c for pair in all_pairs for c in pair)
    preview_labels=[c for c,_ in frequency.most_common(40)]
    if EPS in alphabet and EPS not in preview_labels:
        preview_labels[-1]=EPS
    preview_labels=sorted(preview_labels)
    shown=matrix_df.loc[preview_labels,preview_labels]
    fig,ax=plt.subplots(figsize=(max(9,len(shown)*0.3),max(8,len(shown)*0.3)))
    heat=ax.imshow(shown.values,cmap="Blues")
    ax.set_xticks(range(len(shown)),shown.columns,rotation=90)
    ax.set_yticks(range(len(shown)),shown.index)
    ax.set_xlabel("Predicted character"); ax.set_ylabel("Ground-truth character")
    ax.set_title("OCR character confusion matrix (most frequent 40 characters at most)")
    fig.colorbar(heat,ax=ax); fig.tight_layout()
    fig.savefig(ocr_out/"character_confusion_matrix.png",dpi=180)
    plt.show()
    details_df.to_csv(ocr_out/"ocr_predictions.csv",index=False,encoding="utf-8-sig")
    (ocr_out/"metrics.json").write_text(json.dumps(ocr_summary,indent=2))
    display(details_df.head(20))
    print(json.dumps(ocr_summary,indent=2))
    if unsupported_refs:
        print("WARNING: references contain characters absent from the supplied OCR dictionary; these samples remain in the score.")


## 10. Save and download evaluation results
Missing evaluations are reported explicitly. No tracking metrics are produced. Retain the dataset source/version and check training overlap before reporting these as independent test results.

In [ ]:
import importlib.metadata
summary={"detection":detection_metrics,"ocr":ocr_summary,
         "settings":{"imgsz":EVAL_IMAGE_SIZE,"batch":EVAL_BATCH,"conf_for_AP":AP_MIN_CONF,
                     "nms_iou":NMS_IOU,"max_det":MAX_DETECTIONS},
         "packages":{p:importlib.metadata.version(p) for p in ["ultralytics","paddleocr","paddlepaddle"]},
         "weights_sha256":{p.name:hashlib.sha256(p.read_bytes()).hexdigest()
                            for p in [VEHICLE_WEIGHTS,PLATE_WEIGHTS,OCR_DIR/"inference.pdiparams"]}}
(OUT/"evaluation_summary.json").write_text(json.dumps(summary,indent=2))
print("Detection metrics:"); display(pd.DataFrame(detection_metrics))
print("OCR metrics:"); display(pd.DataFrame([ocr_summary]))
# Exclude copied evaluation images/labels from the result archive.
# Dataset manifests and metadata are retained for reproducibility.
archive_path=Path(str(OUT)+".zip")
with zipfile.ZipFile(archive_path,"w",zipfile.ZIP_DEFLATED) as archive:
    for path in OUT.rglob("*"):
        if not path.is_file(): continue
        relative=path.relative_to(OUT)
        if relative.parts[0]=="prepared" and any(part in {"images","labels"} for part in relative.parts):
            continue
        archive.write(path,relative.as_posix())
files.download(str(archive_path))


## References
[Ultralytics Validation](https://docs.ultralytics.com/modes/val/) · [PaddleOCR TextRecognition](https://www.paddleocr.ai/v3.0.2/en/version3.x/module_usage/text_recognition.html)